In [17]:
import sympy as sp
import pandas as pd

In [18]:
def create_x_y_matrix(x, y, deg):
    n = len(x)
    mat_x = [[0 for _ in range(deg + 1)] for _ in range(n)]

    for j in range(deg + 1):
        for i in range(n):
            mat_x[i][j] = x[i] ** j

    mat_y = y
    return mat_x, mat_y

In [19]:
def read_wine_csv(file_path):
    df = pd.read_csv(file_path)
    # Chuẩn hóa tên cột
    df.columns = df.columns.str.strip().str.replace('"', '')
    
    # Cột đầu tiên 'quality' là y
    y_col = df.columns[0]
    y = df[y_col].tolist()
    
    # 11 cột tiếp theo là các tính chất x
    feature_names = df.columns[1:].tolist()
    
    # Lấy dữ liệu dạng danh sách cho từng cột x
    X_cols = [df[col].tolist() for col in feature_names]
    
    return X_cols, y, feature_names

In [20]:
def copy_matrix(M):
    """
    Tạo bản sao sâu (deep copy) của ma trận 2D.
    """
    return [row[:] for row in M]

In [21]:
def get_submatrix(M, row_to_remove, col_to_remove):
    return [[M[i][j] for j in range(len(M[i])) if j != col_to_remove]
            for i in range(len(M)) if i != row_to_remove]

In [22]:
def determinant(M):
    n = len(M)
    if n == 1:
        return M[0][0]
    if n == 2:
        return M[0][0] * M[1][1] - M[0][1] * M[1][0]

    det = sp.Integer(0)
    for col in range(n):
        sign = (-1) ** col
        sub = get_submatrix(M, 0, col)
        det += sign * M[0][col] * determinant(sub)
    return sp.expand(det)

In [23]:
def identity_matrix(n):
    return [[sp.Integer(1) if i == j else sp.Integer(0) for j in range(n)] for i in range(n)]

In [24]:
def rref(M_augmented):
    A = copy_matrix(M_augmented)
    rows = len(A)
    cols = len(A[0])

    pivot_row = 0
    for col in range(cols):
        swap_row = -1
        for r in range(pivot_row, rows):
            if sp.simplify(A[r][col]) != 0:
                swap_row = r
                break

        if swap_row == -1:
            continue

        A[pivot_row], A[swap_row] = A[swap_row], A[pivot_row]

        pivot_val = A[pivot_row][col]
        A[pivot_row] = [sp.simplify(x / pivot_val) for x in A[pivot_row]]

        for r in range(rows):
            if r != pivot_row:
                factor = A[r][col]
                A[r] = [sp.simplify(item_r - factor * item_p) for item_r, item_p in zip(A[r], A[pivot_row])]

        pivot_row += 1
        if pivot_row >= rows:
            break
    return A

In [ ]:
def invert_matrix(A):
    n = len(A)
    # Tạo ma trận đơn vị I
    I = identity_matrix(n)
    # Ghép ma trận A và I -> [A | I]
    augmented = [A[i] + I[i] for i in range(n)]
    
    # Biến đổi Gauss-Jordan RREF
    eliminated = rref(augmented)
    
    # Lấy nửa bên phải chính là A^(-1)
    inv_A = [row[n:] for row in eliminated]
    return inv_A

In [26]:
def transpose(M):
    return [[M[j][i] for j in range(len(M))] for i in range(len(M[0]))]

In [27]:
def multiply_matrices(A, B):
    rows_A, cols_A = len(A), len(A[0])
    rows_B, cols_B = len(B), len(B[0])
    if cols_A != rows_B:
        raise ValueError("Invalid matrix dimensions for multiplication!")

    C = [[sp.Integer(0) for _ in range(cols_B)] for _ in range(rows_A)]
    for i in range(rows_A):
        for j in range(cols_B):
            C[i][j] = sp.simplify(sum(A[i][k] * B[k][j] for k in range(cols_A)))
    return C

In [ ]:
def calculate_beta(X, y):
    X_float = [[float(val) for val in row] for row in X]
    y_mat = [[float(val)] for val in y]

    X_T = transpose(X_float)

    X_T_X = multiply_matrices(X_T, X_float)

    inv_X_T_X = invert_matrix(X_T_X)

    X_T_y = multiply_matrices(X_T, y_mat)

    beta = multiply_matrices(inv_X_T_X, X_T_y)

    return beta

In [ ]:
def predict(X, theta):
    if (len(X) != len(theta)):
        print("Cannot calculate")
        return None
    res = 0
    for i in range(len(X)):
        res += X[i]*theta[i]

    return res

In [ ]:
# 1. Đọc dữ liệu từ file wine.csv
X_cols, y_data, feature_names = read_wine_csv("wine.csv")

# Thử nghiệm với 50 dòng đầu để tính ma trận ký hiệu bằng SymPy cho nhanh
sample_size = 50
y_sample = y_data[:sample_size]

# 2. Gọi hàm create_x_y_matrix() cho từng cột đặc trưng x với deg=1
# Thu được danh sách các ma trận x_i đơn biến
single_matrices = []
for col in X_cols:
    x_sample = col[:sample_size]
    mat_x_i, _ = create_x_y_matrix(x_sample, y_sample, deg=1)
    single_matrices.append(mat_x_i)

# 3. Ghép thành ma trận X_mat tổng hợp (50 dòng x 12 cột)
n = len(y_sample)
X_mat = []
for i in range(n):
    # Cột 0: Lấy cột số 1 (x^0 = 1) từ ma trận đầu tiên
    row = [sp.sympify(single_matrices[0][i][0])]
    
    # Cột 1 -> 11: Lấy các giá trị đặc trưng (x^1) của từng cột
    for k in range(len(feature_names)):
        row.append(sp.sympify(single_matrices[k][i][1]))
        
    X_mat.append(row)


# 4. Tính 12 hệ số Beta = (X^T * X)^(-1) * X^T * y
beta = calculate_beta(X_mat, y_sample)

# 5. In ra 12 hệ số tương ứng
print("--- KẾT QUẢ TÍNH 12 HỆ SỐ BETA (CÂU A) ---")
print(f"Beta_0 (Hệ số tự do / Intercept) = {sp.N(beta[0][0]):.6f}")

for idx, name in enumerate(feature_names, start=1):
    print(f"Beta_{idx} ({name}) = {sp.N(beta[idx][0]):.6f}")

In [ ]:
def norm():
    return None

In [ ]:
def subtract_vec():
    return None

_IncompleteInputError: incomplete input (2191553988.py, line 1)

In [ ]:
def residual_vector(y_val, y_cup_val):
    return norm(subtract_vec(y_val, y_cup_val))